# Chest X-Ray Binary Classification: Lightweight Prototype
This notebook implements a lightweight classification prototype (MobileNetV3Small) as a fallback for Swin-style experiments, optimized for stability and low memory usage in Colab.

In [15]:
!pip install -q gdown

!gdown --fuzzy "https://drive.google.com/file/d/1GrJHVstisCglttDnZdtfc8_rTIeBiKnz/view?usp=sharing" -O /content/archive.zip

import os
import zipfile
import shutil

zip_path = "/content/archive.zip"
extract_path = "/content/dataset"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction done.")
print("Top-level contents:", os.listdir(extract_path))

Downloading...
From (original): https://drive.google.com/uc?id=1GrJHVstisCglttDnZdtfc8_rTIeBiKnz
From (redirected): https://drive.google.com/uc?id=1GrJHVstisCglttDnZdtfc8_rTIeBiKnz&confirm=t&uuid=dc31a860-6008-4e00-8a9e-b9f31be74e51
To: /content/archive.zip
100% 2.46G/2.46G [00:39<00:00, 62.5MB/s]
Extraction done.
Top-level contents: ['chest_xray']


In [16]:
import os
import shutil

for root, dirs, files in os.walk("/content/dataset"):
    for d in dirs:
        if d in ["__MACOSX", "_MACOSX"]:
            shutil.rmtree(os.path.join(root, d), ignore_errors=True)

DATASET_DIR = "/content/dataset/chest_xray"
TRAIN_PATH = os.path.join(DATASET_DIR, "train")
TEST_PATH = os.path.join(DATASET_DIR, "test")

print("DATASET_DIR:", DATASET_DIR)
print("TRAIN_PATH exists:", os.path.exists(TRAIN_PATH))
print("TEST_PATH exists:", os.path.exists(TEST_PATH))
print("Train classes:", os.listdir(TRAIN_PATH))
print("Test classes:", os.listdir(TEST_PATH))

DATASET_DIR: /content/dataset/chest_xray
TRAIN_PATH exists: True
TEST_PATH exists: True
Train classes: ['NORMAL', 'PNEUMONIA']
Test classes: ['NORMAL', 'PNEUMONIA']


In [17]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils import class_weight

# Configuration
DATASET_DIR = '/content/dataset/chest_xray'
TRAIN_PATH = os.path.join(DATASET_DIR, 'train')
TEST_PATH = os.path.join(DATASET_DIR, 'test')

IMG_SIZE = (224, 224)
BATCH_SIZE = 8
EPOCHS = 5
SEED = 42

In [13]:
# Cell removed - Redundant dataset setup

Please ensure your dataset is organized as follows:
- /content/dataset/chest_xray/train/(NORMAL|PNEUMONIA)
- /content/dataset/chest_xray/test/(NORMAL|PNEUMONIA)


In [11]:
# Cell removed - Redundant dataset setup

Please ensure your dataset is organized as follows:
- /content/dataset/chest_xray/train/(NORMAL|PNEUMONIA)
- /content/dataset/chest_xray/test/(NORMAL|PNEUMONIA)


In [10]:
# Cell removed - Redundant dataset setup

Please ensure your dataset is organized as follows:
- /content/dataset/chest_xray/train/(NORMAL|PNEUMONIA)
- /content/dataset/chest_xray/test/(NORMAL|PNEUMONIA)


In [9]:
# Cell removed - Redundant dataset setup

Please ensure your dataset is organized as follows:
- /content/dataset/chest_xray/train/(NORMAL|PNEUMONIA)
- /content/dataset/chest_xray/test/(NORMAL|PNEUMONIA)


In [8]:
# Cell removed - Redundant dataset setup

Please ensure your dataset is organized as follows:
- /content/dataset/chest_xray/train/(NORMAL|PNEUMONIA)
- /content/dataset/chest_xray/test/(NORMAL|PNEUMONIA)


In [7]:
# Cell removed - Redundant dataset setup

In [18]:
# 2. Load Data with Validation Split
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH,
    shuffle=False,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

class_names = train_ds.class_names
print(f'Classes: {class_names}')

Found 5216 files belonging to 2 classes.
Using 4173 files for training.
Found 5216 files belonging to 2 classes.
Using 1043 files for validation.
Found 624 files belonging to 2 classes.
Classes: ['NORMAL', 'PNEUMONIA']


### Model Construction
We use a Swin Transformer Tiny architecture. Note: For a pure 'smoke test' or if specific libraries are missing, we use a ResNet-style backbone with a Transformer-like head to ensure immediate runability, but here we define the standard Swin approach.

In [19]:
# Lightweight fallback architecture (MobileNetV3Small)
# MobileNetV3 internally handles specialized preprocessing; manual rescaling is removed.
base_model = tf.keras.applications.MobileNetV3Small(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,013,105 (3.86 MB)

 Trainable params: 73,985 (289.00 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [20]:
# 3. Calculate Class Weights & Define Callbacks
y_train = np.concatenate([y for x, y in train_ds], axis=0)
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train.flatten())
class_weight_dict = dict(enumerate(weights))

model_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    callbacks.ModelCheckpoint('best_model.keras', save_best_only=True)
]

# 4. Training
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=model_callbacks
)

Epoch 1/5
522/522 ━━━━━━━━━━━━━━━━━━━━ 60s 107ms/step - accuracy: 0.9260 - auc: 0.9816 - loss: 0.1712 - val_accuracy: 0.9712 - val_auc: 0.9921 - val_loss: 0.0968
Epoch 2/5
522/522 ━━━━━━━━━━━━━━━━━━━━ 66s 126ms/step - accuracy: 0.9569 - auc: 0.9921 - loss: 0.1068 - val_accuracy: 0.9645 - val_auc: 0.9931 - val_loss: 0.0954
Epoch 3/5
522/522 ━━━━━━━━━━━━━━━━━━━━ 54s 104ms/step - accuracy: 0.9636 - auc: 0.9948 - loss: 0.0880 - val_accuracy: 0.9799 - val_auc: 0.9945 - val_loss: 0.0710
Epoch 4/5
522/522 ━━━━━━━━━━━━━━━━━━━━ 54s 104ms/step - accuracy: 0.9693 - auc: 0.9958 - loss: 0.0772 - val_accuracy: 0.9396 - val_auc: 0.9928 - val_loss: 0.1754
Epoch 5/5
522/522 ━━━━━━━━━━━━━━━━━━━━ 56s 106ms/step - accuracy: 0.9717 - auc: 0.9958 - loss: 0.0715 - val_accuracy: 0.9664 - val_auc: 0.9955 - val_loss: 0.0709


In [21]:
# Evaluation and Predictions
y_true = np.concatenate([y for x, y in test_ds], axis=0)
predictions = model.predict(test_ds)
y_pred = (predictions > 0.5).astype(int)

# Get Relative Paths for robustness
file_paths = test_ds.file_paths
rel_filenames = [os.path.relpath(f, TEST_PATH) for f in file_paths]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

auc_score = roc_auc_score(y_true, predictions)
print(f"ROC-AUC Score: {auc_score:.4f}")

78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 83ms/step

Classification Report:
              precision    recall  f1-score   support

      NORMAL       1.00      0.43      0.60       234
   PNEUMONIA       0.75      1.00      0.85       390

    accuracy                           0.79       624
   macro avg       0.87      0.72      0.73       624
weighted avg       0.84      0.79      0.76       624

ROC-AUC Score: 0.9619


In [22]:
# Export Results to CSV
results_df = pd.DataFrame({
    'filename': rel_filenames,
    'true_label': y_true.flatten().astype(int),
    'pred_label': y_pred.flatten(),
    'prob_pneumonia': predictions.flatten()
})

results_df.to_csv('swin_test_predictions.csv', index=False)
print("Exported swin_test_predictions.csv successfully.")
display(results_df.head())

Exported swin_test_predictions.csv successfully.


,filename,true_label,pred_label,prob_pneumonia
0,NORMAL/IM-0001-0001.jpeg,0,0,0.240322
1,NORMAL/IM-0003-0001.jpeg,0,0,0.032218
2,NORMAL/IM-0005-0001.jpeg,0,0,0.344182
3,NORMAL/IM-0006-0001.jpeg,0,0,0.121314
4,NORMAL/IM-0007-0001.jpeg,0,0,0.422677
